# Segmentación de Clientes (RFM)

Proyecto de clustering listo para **Google Colab**.

**Pregunta:** ¿podemos segmentar clientes según Recencia, Frecuencia y Monto?

**Método:** K-Means sobre variables RFM.

**Stack:** pandas, seaborn, scikit-learn

Dataset de ejemplo para segmentación RFM.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar / simular datos RFM

In [ ]:
np.random.seed(RANDOM_STATE)
n = 500
df = pd.DataFrame({
    "Recency": np.random.randint(1, 365, n),
    "Frequency": np.random.randint(1, 20, n),
    "Monetary": np.random.uniform(10, 500, n).round(2),
})
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df["Recency"], kde=True, ax=axes[0])
axes[0].set_title("Recency")
sns.histplot(df["Frequency"], kde=True, ax=axes[1])
axes[1].set_title("Frequency")
sns.histplot(df["Monetary"], kde=True, ax=axes[2])
axes[2].set_title("Monetary")
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, height=2.5)
plt.suptitle("Pairplot RFM", y=1.02)
plt.show()

### Hallazgos
- Distribuciones típicas de RFM.
- Ideal para clustering no supervisado.

## 3. Preprocesamiento

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print(X_scaled.shape)

## 4. Elegir número de clusters (Elbow + Silhouette)

In [ ]:
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K, inertias, marker="o")
axes[0].set_title("Elbow")
axes[0].set_xlabel("k")
axes[1].plot(K, silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

## 5. Clustering final

In [ ]:
k_final = 4
km = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = km.fit_predict(X_scaled)
print(df["cluster"].value_counts())

In [ ]:
centros = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_), columns=df.columns[:-1])
print("Centros de los clusters:")
centros.round(1)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Frequency", y="Monetary", hue="cluster", palette="tab10")
plt.title("Clusters RFM")
plt.show()

## 6. Interpretación de segmentos

In [ ]:
perfil = df.groupby("cluster")[["Recency", "Frequency", "Monetary"]].mean().round(1)
perfil["n"] = df["cluster"].value_counts()
perfil.sort_values("Monetary", ascending=False)

## 7. Conclusiones

- Segmentos típicos RFM: Champions, Loyal, At Risk, Hibernating.
- El número óptimo suele estar entre 3–5 clusters.
- Se puede extender a datos reales de transacciones.

**Siguiente paso:** asignar nombres de negocio a cada segmento.